In [1]:
import sys
import os

def get_UGCE_directory():
    """Get the path of the 'UGCE-User-Guided-Counterfactual-Exploration' directory."""
    current_dir = os.getcwd()
    target_dir = 'UGCE-User-Guided-Counterfactual-Exploration'
    
    while os.path.basename(current_dir) != target_dir:
        current_dir = os.path.dirname(current_dir)
        if current_dir == os.path.dirname(current_dir):
            return None
        
    return current_dir

def get_system_slash():
    """Get the system-specific directory separator."""
    return os.sep

UGCE_dir = get_UGCE_directory()
sys.path.append(UGCE_dir)
sep = get_system_slash()
sys.path.append(UGCE_dir + get_system_slash() + 'src')

from dataLoader import *
from utils import *

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
seed_number = 42
import random

random.seed(seed_number)
np.random.seed(seed_number)

In [4]:
datasetName = 'GermanCredit'

In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
import pandas as pd
import dice_ml
from dice_ml.utils import helpers

from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier

names = ['existingchecking', 'duration', 'credithistory', 'purpose', 'creditamount', 
         'savings', 'employmentsince', 'installmentrate', 'statussex', 'otherdebtors', 
         'residencesince', 'property', 'age', 'otherinstallmentplans', 'housing', 
         'existingcredits', 'job', 'peopleliable', 'telephone', 'foreignworker', 'target']
dataset = pd.read_csv(f"{ugce_dir}/data/GermanCredit.data", sep=' ', header=None ,names = names)
TARGET_COLUMN = 'target'
dataset[TARGET_COLUMN] = LabelEncoder().fit_transform(dataset[TARGET_COLUMN])
datasetX = dataset.drop(columns=[TARGET_COLUMN])
target = dataset[TARGET_COLUMN]

x_train, x_test, y_train, y_test = train_test_split(datasetX,
                                                    target,
                                                    test_size=0.2,
                                                    random_state=0,
                                                    stratify=target)

numerical = datasetX.select_dtypes(include=[np.number]).columns.tolist()
categorical = x_train.columns.difference(numerical)

numeric_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())])

categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore'))])

transformations = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numerical),
        ('cat', categorical_transformer, categorical)])

model = RandomForestClassifier(random_state=42)

model = Pipeline(steps=[('preprocessor', transformations),
                      ('classifier', model)])

model.fit(x_train, y_train)

y_pred = model.predict(x_test)
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy: ", accuracy)

negative_instances = x_test[model.predict(x_test) == 0]
instances_to_explain = negative_instances
print("Number of instances to explain: ", len(instances_to_explain))

Accuracy:  0.75
Number of instances to explain:  166


In [6]:
from ugce import *

iea = UGCE(model, datasetX,
           numerical, categorical)

iea.dataset.describe()

,existingchecking,duration,credithistory,purpose,creditamount,savings,employmentsince,installmentrate,statussex,otherdebtors,residencesince,property,age,otherinstallmentplans,housing,existingcredits,job,peopleliable,telephone,foreignworker
count,1000.000000,1000.000000,1000.00000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.00000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000
mean,1.577000,20.903000,2.54500,3.277000,3271.258000,1.105000,2.384000,2.973000,1.68200,0.145000,2.845000,1.358000,35.546000,1.675000,0.929000,1.407000,1.904000,1.155000,0.404000,0.037000
std,1.257638,12.058814,1.08312,2.739302,2822.736876,1.580023,1.208306,1.118715,0.70808,0.477706,1.103718,1.050209,11.375469,0.705601,0.531264,0.577654,0.653614,0.362086,0.490943,0.188856
min,0.000000,4.000000,0.00000,0.000000,250.000000,0.000000,0.000000,1.000000,0.00000,0.000000,1.000000,0.000000,19.000000,0.000000,0.000000,1.000000,0.000000,1.000000,0.000000,0.000000
25%,0.000000,12.000000,2.00000,1.000000,1365.500000,0.000000,2.000000,2.000000,1.00000,0.000000,2.000000,0.000000,27.000000,2.000000,1.000000,1.000000,2.000000,1.000000,0.000000,0.000000
50%,1.000000,18.000000,2.00000,3.000000,2319.500000,0.000000,2.000000,3.000000,2.00000,0.000000,3.000000,1.000000,33.000000,2.000000,1.000000,1.000000,2.000000,1.000000,0.000000,0.000000
75%,3.000000,24.000000,4.00000,4.000000,3972.250000,2.000000,4.000000,4.000000,2.00000,0.000000,4.000000,2.000000,42.000000,2.000000,1.000000,2.000000,2.000000,1.000000,1.000000,0.000000
max,3.000000,72.000000,4.00000,9.000000,18424.000000,4.000000,4.000000,4.000000,3.00000,2.000000,4.000000,3.000000,75.000000,2.000000,2.000000,4.000000,3.000000,2.000000,1.000000,1.000000


In [7]:
pd.set_option('display.max_columns', None)
numerical_columns = iea.dataset.select_dtypes(include=['int64', 'float64']).columns

non_zero_descriptions = {}

for col in numerical_columns:
    non_zero_values = iea.dataset[iea.dataset[col] != 0][col]
    if not non_zero_values.empty:
        non_zero_descriptions[col] = non_zero_values.describe()

# Display results
for feature, stats in non_zero_descriptions.items():
    print(f"\n Feature: {feature}")
    print(stats)


 Feature: existingchecking
count    726.000000
mean       2.172176
std        0.940637
min        1.000000
25%        1.000000
50%        3.000000
75%        3.000000
max        3.000000
Name: existingchecking, dtype: float64

 Feature: duration
count    1000.000000
mean       20.903000
std        12.058814
min         4.000000
25%        12.000000
50%        18.000000
75%        24.000000
max        72.000000
Name: duration, dtype: float64

 Feature: credithistory
count    960.000000
mean       2.651042
std        0.969880
min        1.000000
25%        2.000000
50%        2.000000
75%        4.000000
max        4.000000
Name: credithistory, dtype: float64

 Feature: purpose
count    766.000000
mean       4.278068
std        2.347512
min        1.000000
25%        3.000000
50%        4.000000
75%        4.000000
max        9.000000
Name: purpose, dtype: float64

 Feature: creditamount
count     1000.000000
mean      3271.258000
std       2822.736876
min        250.000000
25%       13

In [ ]:
feat_unique = {}
for col in datasetX.columns:
    feat_unique[col] = len(datasetX[col].unique())
feat_unique
## sort the features by the number of unique values
top_3_difficult_to_change_cols = sorted(feat_unique.items(), key=lambda x: x[1])
top_3_difficult_to_change_cols

[('peopleliable', 2),
 ('telephone', 2),
 ('foreignworker', 2),
 ('otherdebtors', 3),
 ('otherinstallmentplans', 3),
 ('housing', 3),
 ('existingchecking', 4),
 ('installmentrate', 4),
 ('statussex', 4),
 ('residencesince', 4),
 ('property', 4),
 ('existingcredits', 4),
 ('job', 4),
 ('credithistory', 5),
 ('savings', 5),
 ('employmentsince', 5),
 ('purpose', 10),
 ('duration', 33),
 ('age', 53),
 ('creditamount', 921)]

# Constraints Type Series:
1. Immutability
2. Ranges
3. Directionality

In [ ]:
from ugce import *

iea = UGCE(model, datasetX,
           numerical, categorical)

updated_constraints = {
    1: {
        'foreignworker': 'i',
        'peopleliable': 'i'
    },
    2:{
        'foreignworker': 'i',
        'peopleliable': 'i',
        'creditamount': (250, 1500),
        'duration': (4, 20),
        
    },
    3: {
        'foreignworker': 'i',
        'peopleliable': 'i',
        'creditamount': (250, 1500),
        'duration': (4, 20),
        "age": 'incr',
        "installmentrate": 'incr'
    }
}
for key in list(updated_constraints.keys()):
    values = updated_constraints[key]
    for col in iea.feature_names:
        if col not in values:
            updated_constraints[key][col] = ''

results_incremental_imm_ranges_direct_arr = []
import time
strategy = "fix_population_update_fitness"
for i in range(5):
    results_incremental_imm_ranges_direct = iea.explain_instances(instances_to_explain, seed_number=None,
        dynamic_constraints=True, constraints={},
        initial_population_variability=0.9, data_distribution=True,
        strategy=strategy, population_size_dynamic=0, cfes_requested=2,
        num_generations=500, initial_population_strategy="kdtree", population_size=10, early_stopping_iterations=5,
        num_parents=19, selection_method="sus", tournsize=5, crossover_points=2,
        elite_ratio=0.1, cxpb=0.8, mutpb=0.7, regeneration_tries_intermediate=0,
        regeneration_tries_intermediate_after_updating_constraints=0,
        updated_constraints=updated_constraints, automatic_user_acceptance=True,
        verbose=False, running_times_per_instance=1,
        distance_metric="weighted_l1", multistep_updated_constraints=True,
        normalized_fitness=True, reweighting_lambdas_after_generations=-1,
        initial_lambdas_without_constraints={"lambda1":0.2, "lambda2":0.2, "lambda3":1},
        initial_lambdas_with_constraints={"lambda1":0.2, "lambda2":0.2, "lambda3":1},
        lambdas_after_updating_constraints={"lambda1":0.2, "lambda2":0.2, "lambda3":1},
        lambdas_reweighting_after_updating_constraints_and_some_generations={"lambda1":0.2, "lambda2":0.2, "lambda3":1})
    results_incremental_imm_ranges_direct_arr.append(results_incremental_imm_ranges_direct)
import pickle
results_dir = f'{UGCE_dir}/results/assess_ugce/{datasetName}/incremental'
os.makedirs(results_dir, exist_ok=True)
pickle.dump(results_incremental_imm_ranges_direct, open(f"{results_dir}/results_incremental_imm_ranges_direct_arr.pkl", "wb"))

  0%|          | 0/166 [00:00<?, ?it/s]

100%|██████████| 166/166 [01:49<00:00,  1.52it/s]


Empty intermediate counter: 0


100%|██████████| 166/166 [01:48<00:00,  1.53it/s]


Empty intermediate counter: 0


...

Empty intermediate counter: 0


100%|██████████| 166/166 [01:48<00:00,  1.53it/s]


Empty intermediate counter: 0


100%|██████████| 166/166 [01:47<00:00,  1.54it/s]

Empty intermediate counter: 0


In [ ]:
strategy = "fix_population_update_fitness"

In [ ]:
## load the results
import pickle
results_dir = f'{UGCE_dir}/results/assess_ugce/{datasetName}/incremental'
results_incremental_imm_ranges_direct = pickle.load(open(f'{results_dir}/results_incremental{strategy}_ranges_imm_direct.pkl', 'rb'))

# Constraints Type Series:
1. Ranges
2. Immutability
3. Directionality

In [ ]:
from ugce import *

iea = UGCE(model, datasetX,
           numerical, categorical)

updated_constraints = {
    1: {
        'creditamount': (250, 1500),
        'duration': (4, 20)
    },
    2:{
        'creditamount': (250, 1500),
        'duration': (4, 20),
        'foreignworker': 'i',
        'peopleliable': 'i'
    },
    3: {
       'creditamount': (250, 1500),
        'duration': (4, 20),
        'foreignworker': 'i',
        'peopleliable': 'i',
        "age": 'incr',
        "installmentrate": 'incr'
    }
}
for key in list(updated_constraints.keys()):
    values = updated_constraints[key]
    for col in iea.feature_names:
        if col not in values:
            updated_constraints[key][col] = ''

results_incremental_ranges_imm_incr_arr = []
import time
strategy = "fix_population_update_fitness"
for i in range(5):
    results_incremental_ranges_imm_incr = iea.explain_instances(instances_to_explain, seed_number=None,
        dynamic_constraints=True, constraints={},
        initial_population_variability=0.9, data_distribution=True,
        strategy=strategy, population_size_dynamic=0, cfes_requested=2,
        num_generations=500, initial_population_strategy="kdtree", population_size=10, early_stopping_iterations=5,
        num_parents=19, selection_method="sus", tournsize=5, crossover_points=2,
        elite_ratio=0.1, cxpb=0.8, mutpb=0.7, regeneration_tries_intermediate=0,
        regeneration_tries_intermediate_after_updating_constraints=0,
        updated_constraints=updated_constraints, automatic_user_acceptance=True,
        verbose=False, running_times_per_instance=1,
        distance_metric="weighted_l1", multistep_updated_constraints=True,
        normalized_fitness=True, reweighting_lambdas_after_generations=-1,
        initial_lambdas_without_constraints={"lambda1":0.2, "lambda2":0.2, "lambda3":1},
        initial_lambdas_with_constraints={"lambda1":0.2, "lambda2":0.2, "lambda3":1},
        lambdas_after_updating_constraints={"lambda1":0.2, "lambda2":0.2, "lambda3":1},
        lambdas_reweighting_after_updating_constraints_and_some_generations={"lambda1":0.2, "lambda2":0.2, "lambda3":1})
    results_incremental_ranges_imm_incr_arr.append(results_incremental_ranges_imm_incr)
import pickle
results_dir = f'{UGCE_dir}/results/assess_ugce/{datasetName}/incremental'
os.makedirs(results_dir, exist_ok=True)
pickle.dump(results_incremental_ranges_imm_incr, open(f"{results_dir}/results_incremental_ranges_imm_incr_arr.pkl", "wb"))

  0%|          | 0/166 [00:00<?, ?it/s]

100%|██████████| 166/166 [01:43<00:00,  1.60it/s]


Empty intermediate counter: 0


100%|██████████| 166/166 [01:44<00:00,  1.60it/s]


Empty intermediate counter: 0


...

Empty intermediate counter: 0


100%|██████████| 166/166 [01:44<00:00,  1.60it/s]


Empty intermediate counter: 0


100%|██████████| 166/166 [01:44<00:00,  1.59it/s]

Empty intermediate counter: 0


In [18]:
## load the results
import pickle
results_dir = f'{UGCE_dir}/results/assess_ugce/{datasetName}/incremental'
results_incremental_ranges_imm_direct = pickle.load(open(f'{results_dir}/results_incremental{strategy}_ranges_imm_direct.pkl', 'rb'))

# Constraints Type Series:
1. Directionality
2. Immutability
3. Ranges

In [ ]:
from ugce import *

iea = UGCE(model, datasetX,
           numerical, categorical)

updated_constraints = {
    1: {
        "age": 'incr',
        "installmentrate": 'incr'
    },
    2:{
        "age": 'incr',
        "installmentrate": 'incr',
        'foreignworker': 'i',
        'peopleliable': 'i'
    },
    3: {
        "age": 'incr',
        "installmentrate": 'incr',
        'foreignworker': 'i',
        'peopleliable': 'i',
        'creditamount': (250, 1500),
        'duration': (4, 20)
    }
}
for key in list(updated_constraints.keys()):
    values = updated_constraints[key]
    for col in iea.feature_names:
        if col not in values:
            updated_constraints[key][col] = ''

import time
strategy = "fix_population_update_fitness"
results_incremental_dir_im_range_arr = []
for i in range(5):
    results_incremental_dir_im_range = iea.explain_instances(instances_to_explain, seed_number=None,
        dynamic_constraints=True, constraints={},
        initial_population_variability=0.9, data_distribution=True,
        strategy=strategy, population_size_dynamic=0, cfes_requested=2,
        num_generations=500, initial_population_strategy="kdtree", population_size=10, early_stopping_iterations=5,
        num_parents=19, selection_method="sus", tournsize=5, crossover_points=2,
        elite_ratio=0.1, cxpb=0.8, mutpb=0.7, regeneration_tries_intermediate=0,
        regeneration_tries_intermediate_after_updating_constraints=0,
        updated_constraints=updated_constraints, automatic_user_acceptance=True,
        verbose=False, running_times_per_instance=1,
        distance_metric="weighted_l1", multistep_updated_constraints=True,
        normalized_fitness=True, reweighting_lambdas_after_generations=-1,
        initial_lambdas_without_constraints={"lambda1":0.2, "lambda2":0.2, "lambda3":1},
        initial_lambdas_with_constraints={"lambda1":0.2, "lambda2":0.2, "lambda3":1},
        lambdas_after_updating_constraints={"lambda1":0.2, "lambda2":0.2, "lambda3":1},
        lambdas_reweighting_after_updating_constraints_and_some_generations={"lambda1":0.2, "lambda2":0.2, "lambda3":1})
    results_incremental_dir_im_range_arr.append(results_incremental_dir_im_range)
import pickle
results_dir = f'{UGCE_dir}/results/assess_ugce/{datasetName}/incremental'
os.makedirs(results_dir, exist_ok=True)
pickle.dump(results_incremental_dir_im_range, open(f"{results_dir}/results_incremental_dir_im_range_arr.pkl", "wb"))

  0%|          | 0/166 [00:00<?, ?it/s]

100%|██████████| 166/166 [01:51<00:00,  1.49it/s]


Empty intermediate counter: 0


100%|██████████| 166/166 [01:52<00:00,  1.48it/s]


Empty intermediate counter: 0


...

Empty intermediate counter: 0


100%|██████████| 166/166 [01:52<00:00,  1.47it/s]


Empty intermediate counter: 0


100%|██████████| 166/166 [01:52<00:00,  1.48it/s]

Empty intermediate counter: 0


In [85]:
## load the results
import pickle
results_dir = f'{UGCE_dir}/results/assess_ugce/{datasetName}/incremental'
results_incremental_dir_im_range = pickle.load(open(f'{results_dir}/results_incremental{strategy}_ranges_imm_direct.pkl', 'rb'))

# Make Plots

In [ ]:
from test_utils import gather_results_sequence_of_type_constraints
import matplotlib.pyplot as plt

constraint_orders = ["I→R→D", "R→I→D", "D→I→R"]

time_dynamic_imm_ranges_direct, avg_generations_imm_ranges_direct, avg_cfes_found_imm_ranges_direct, avg_proximity_loss_imm_ranges_direct, avg_sparsity_imm_ranges_direct, avg_intermediate_imm_ranges_incr, \
time_dynamic_ranges_imm_incr, avg_generations_ranges_imm_incr, avg_cfes_found_ranges_imm_incr, avg_proximity_loss_ranges_imm_incr, avg_sparsity_ranges_imm_incr, avg_intermediate_ranges_imm_incr, \
time_dynamic_dir_im_range, avg_generations_dir_im_range, avg_cfes_found_dir_im_range, avg_proximity_loss_dir_im_range, avg_sparsity_dir_im_range, avg_intermediate_dir_im_range =\
    gather_results_sequence_of_type_constraints(iea, results_incremental_imm_ranges_direct_arr, results_incremental_ranges_imm_incr_arr, results_incremental_dir_im_range_arr, verbose=True) 

cfe_found = [
        avg_cfes_found_imm_ranges_direct,
        avg_cfes_found_ranges_imm_incr,
        avg_cfes_found_dir_im_range
]
avg_time = [
    time_dynamic_imm_ranges_direct,
    time_dynamic_ranges_imm_incr,
    time_dynamic_dir_im_range
]
avg_weighted_l1 = [
    avg_proximity_loss_imm_ranges_direct,
    avg_proximity_loss_ranges_imm_incr,
    avg_proximity_loss_dir_im_range
]

avg_sparsity = [
    avg_sparsity_imm_ranges_direct,
    avg_sparsity_ranges_imm_incr,
    avg_sparsity_dir_im_range
]
results = {
    "cfe_found": cfe_found,
    "avg_time": avg_time,
    "avg_weighted_l1": avg_weighted_l1,
    "avg_sparsity": avg_sparsity
}

In [ ]:
table_data = pd.DataFrame(results, index=constraint_orders)
print(table_data.to_latex(float_format="%.4f"))

\begin{tabular}{lrrrr}
\toprule
 & cfe_found & avg_time & avg_weighted_l1 & avg_sparsity \\
\midrule
I→R→D & 98.8095 & 1.2945 & 0.0436 & 0.0244 \\
R→I→D & 98.8095 & 1.2231 & 0.0445 & 0.0246 \\
D→I→R & 98.8095 & 1.3567 & 0.0398 & 0.0242 \\
\bottomrule
\end{tabular}

